<a href="https://colab.research.google.com/github/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_6/lessons/lesson_51_final_project/note_lesson_51_final_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧭 Урок 51 — Робота над фінальним проєктом: аудит

| Крок | Що робимо |
|---|---|
| 0 | граф імпортів `news_hub` через `ast` (без запуску проєкту) |
| 1 | fan-in / fan-out: фундамент і точки входу (вправа 1) |
| 2 | цикли імпортів: алгоритм Тар'яна крок за кроком (вправа 2) |
| 3 | правило шарів: хто кому не має права імпортувати (вправа 3) |
| 4 | база: індекс працює лише для свого виразу; зайві індекси (вправа 4) |
| 5 | пагінація OFFSET і keyset (вправа 5) |
| 6 | стан і репліки: throttle у пам'яті процесу (вправа 6) |
| 7 | чек-лист фінального проєкту (вправа 7) |

**Як працювати:** зверху вниз; перед **🔮 Прогнозом** спершу відповідай сам.

Урок 51 — зріз усього курсу: ми міряємо два проєкти курсу, а потім ті самі інструменти застосовуєш до свого фінального проєкту. У книзі — справжні `EXPLAIN ANALYZE` на PostgreSQL з 302 400 новинами; тут — те саме на SQLite, яка є в кожному Python. Повний розбір: [Урок 51](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m6/lesson_51/).

---
## 0. Граф імпортів

`architecture_audit.py` з папки уроку читає кожен `.py` пакета через `ast` і збирає, хто кого імпортує. Проєкт не треба встановлювати: код лише розбирається, не виконується.

In [ ]:
import os
import subprocess
import sys

REPO = "https://github.com/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026.git"
LESSON = "module_6/lessons/lesson_51_final_project"
PROJECTS = "module_5/lessons/lesson_50_ci_cd"
if not os.path.isdir(LESSON):                       # Colab: беремо з GitHub лише потрібні папки
    if not os.path.isdir("course_repo"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse", REPO, "course_repo"], check=True)
        subprocess.run(["git", "-C", "course_repo", "sparse-checkout", "set", LESSON, PROJECTS], check=True)
    os.chdir("course_repo")
sys.path.insert(0, LESSON)

from collections import defaultdict
from pathlib import Path

from architecture_audit import collect, cycles

NEWS = Path(PROJECTS) / "news_hub" / "news_hub"
graph, lines = collect([NEWS])
print(len(graph), "модулів,", sum(lines.values()), "рядків")
print("repository імпортує:", sorted(graph["news_hub.repository"]))

🔮 **Прогноз:** який модуль `news_hub` імпортує найбільше інших модулів проєкту?

<details>
<summary>Відповідь</summary>

`news_hub.api` — 18 модулів: кожен ендпоінт тягне свою частину системи. Це точка входу: її ніхто не імпортує, а вона — всіх.
</details>

---
## 1. Fan-in і fan-out

- **fan-out** модуля — скільки модулів він імпортує (`len(graph[m])`);
- **fan-in** — скільки модулів імпортують його.

Високий fan-in і нульовий fan-out — фундамент (від нього залежать усі, він — ні від кого). Нульовий fan-in — точка входу.

### Вправа 1

Напиши `fan_in(graph)` → словник `{модуль: скільки модулів його імпортують}` (для кожного модуля графа, навіть з нулем). Потім збери `foundation` — модулі з fan-in ≥ 4 і fan-out 0.

In [ ]:
def fan_in(graph: dict[str, set[str]]) -> dict[str, int]:
    # YOUR CODE HERE
    # BEGIN SOLUTION
    counts = {name: 0 for name in graph}
    for targets in graph.values():
        for target in targets:
            counts[target] += 1
    return counts
    # END SOLUTION


incoming = fan_in(graph)
# YOUR CODE HERE
# BEGIN SOLUTION
foundation = {m for m in graph if incoming[m] >= 4 and not graph[m]}
# END SOLUTION

for name in sorted(graph, key=lambda m: -incoming[m])[:6]:
    print(f"{name:<22} fan-in {incoming[name]}  fan-out {len(graph[name])}")
print("фундамент:", sorted(foundation))
assert incoming["news_hub.llm"] == 7 and incoming["news_hub.api"] == 0
assert foundation == {"news_hub.llm", "news_hub.cache", "news_hub.db", "news_hub.parser"}
print("✅ Вправа 1 пройдена")

---
## 2. Цикли імпортів

Цикл — модулі, які через ланцюжок імпортів залежать самі від себе: `A → B → C → A`. Python такий пакет часто не імпортує (`ImportError: … partially initialized module`), а якщо й імпортує — лише доки ніхто не змінить порядок рядків.

`cycles(graph)` шукає **сильно зв'язані компоненти** алгоритмом Тар'яна: один обхід у глибину, і кожен модуль отримує два числа:

- `index` — порядковий номер, коли обхід уперше до нього дійшов;
- `low` — найменший `index`, до якого можна «повернутися» з нього по ребрах, що ведуть у модулі, які ще на стеку.

Коли обхід виходить з модуля, у якого `low == index`, — цей модуль корінь компоненти: знімаємо зі стеку все до нього включно. Понад один модуль — цикл.

Приклад з тестів уроку: `orders → users → payments → orders`, а `utils → orders`.

```mermaid
flowchart TD
    classDef step     fill:#263238,stroke:#90a4ae,color:#ffffff;
    classDef decision fill:#37474f,stroke:#64b5f6,color:#ffffff;
    classDef success  fill:#1b5e20,stroke:#4CAF50,color:#ffffff;
    classDef error    fill:#4e1f1f,stroke:#f44336,color:#ffffff;
    classDef warning  fill:#4a3b00,stroke:#ff9800,color:#ffffff;

    subgraph S1["крок 1: заходимо в orders"]
        direction LR
        A1["orders<br>index 0, low 0"] --> A2["стек: orders"]
    end
    subgraph S2["крок 2: orders → users"]
        direction LR
        B1["users<br>index 1, low 1"] --> B2["стек: orders, users"]
    end
    subgraph S3["крок 3: users → payments"]
        direction LR
        C1["payments<br>index 2, low 2"] --> C2["стек: orders, users, payments"]
    end
    subgraph S4["крок 4: payments → orders — orders на стеку"]
        direction LR
        D1["low[payments] = index[orders] = 0"]
    end
    subgraph S5["крок 5: повертаємось"]
        direction LR
        E1["low[users] = min(1, 0) = 0"] --> E2["low[orders] = 0 = index[orders]"]
    end
    subgraph S6["крок 6: orders — корінь"]
        direction LR
        F1["знімаємо payments, users, orders"] --> F2["цикл із 3 модулів"]
    end
    subgraph S7["крок 7: utils"]
        direction LR
        G1["utils<br>index 3, low 3"] --> G2["utils → orders: вже не на стеку"] --> G3["компонента з 1 модуля:<br>не цикл"]
    end
    S1 --> S2 --> S3 --> S4 --> S5 --> S6 --> S7

    class A1,B1,C1,G1 step
    class A2,B2,C2 step
    class D1,E1 warning
    class E2 decision
    class F1 warning
    class F2 error
    class G2 step
    class G3 success
```

In [ ]:
shop = {
    "orders": {"users"},
    "users": {"payments"},
    "payments": {"orders"},
    "utils": {"orders"},
}
print("цикли:", cycles(shop))
print("news_hub:", cycles(graph) or "немає")

### Вправа 2

У копії графа `news_hub` додай ребро `news_hub.db → news_hub.repository` (ніби в `db.py` з'явився рядок `from .repository import NewsRepository`) і знайди цикли. Збережи результат у `found`.

🔮 **Прогноз:** скільки модулів буде в циклі — два (`db`, `repository`) чи більше?

In [ ]:
broken = {name: set(targets) for name, targets in graph.items()}     # копія: graph не чіпаємо
# YOUR CODE HERE
# BEGIN SOLUTION
broken["news_hub.db"].add("news_hub.repository")
found = cycles(broken)
# END SOLUTION

print(found)
assert found == [["news_hub.db", "news_hub.repository", "news_hub.tables"]]
assert cycles(graph) == []
print("✅ Вправа 2 пройдена")

<details>
<summary>Відповідь</summary>

Три: `repository` імпортує `tables`, а `tables` — `db`. Тож з `db` можна дійти до `repository` і назад двома шляхами, і `tables` — частина циклу. У справжньому проєкті це закінчується так:

`ImportError: cannot import name 'Base' from partially initialized module 'news_hub.db' (most likely due to a circular import)`
</details>

---
## 3. Правило шарів

Цикли ловлять, коли залежності пішли по колу. Але архітектуру ламає й одне ребро «не в той бік»: фундамент, що імпортує точку входу. Домовимось про три шари `news_hub`:

```mermaid
graph LR
    classDef step     fill:#263238,stroke:#90a4ae,color:#ffffff;
    classDef decision fill:#37474f,stroke:#64b5f6,color:#ffffff;
    classDef success  fill:#1b5e20,stroke:#4CAF50,color:#ffffff;
    classDef error    fill:#4e1f1f,stroke:#f44336,color:#ffffff;
    classDef warning  fill:#4a3b00,stroke:#ff9800,color:#ffffff;

    E["2 · точки входу<br>api, jobs, bot"] --> L["1 · логіка<br>repository, analysis, notify, scraper, rss, middleware"]
    L --> B["0 · фундамент<br>db, tables, models, parser, llm, cache, security…"]

    class E warning
    class L step
    class B success
```

Правило: модуль може імпортувати лише свій шар або нижчий.

### Вправа 3

Напиши `violations(graph, layer)` → множину ребер `(звідки, куди)`, де `layer[куди] > layer[звідки]`.

In [ ]:
ENTRY = {"news_hub", "news_hub.api", "news_hub.jobs", "news_hub.bot", "news_hub.bot.__main__",
         "news_hub.bot.factory", "news_hub.bot.handlers", "news_hub.bot.middlewares"}
LOGIC = {"news_hub.repository", "news_hub.analysis", "news_hub.notify", "news_hub.scraper", "news_hub.rss",
         "news_hub.middleware"}
layer = {name: 2 if name in ENTRY else 1 if name in LOGIC else 0 for name in graph}


def violations(graph: dict[str, set[str]], layer: dict[str, int]) -> set[tuple[str, str]]:
    # YOUR CODE HERE
    # BEGIN SOLUTION
    return {(source, target) for source, targets in graph.items() for target in targets
            if layer[target] > layer[source]}
    # END SOLUTION


print("базові модулі:", sorted(n.removeprefix("news_hub.") for n in layer if layer[n] == 0))
print("порушення в news_hub:", violations(graph, layer) or "немає")
shortcut = {name: set(targets) for name, targets in graph.items()}
shortcut["news_hub.cache"].add("news_hub.api")                      # «швидко взяти settings з api»
print("з ярликом:", violations(shortcut, layer))
assert violations(graph, layer) == set()
assert violations(shortcut, layer) == {("news_hub.cache", "news_hub.api")}
print("✅ Вправа 3 пройдена")

Так само влаштований `tests_architecture.py` проєкту нотаток (урок 44): тест розбирає код через `ast` і падає, якщо правило порушено. Такий тест — дешевий спосіб зберегти архітектуру фінального проєкту, коли код росте.

---
## 4. База: індекс і вираз

У книзі — головна знахідка аудиту бази `news_hub`: індекс trigram на `title` існував, займав 59 MB і **не використовувався**, бо запит `search()` порівнює `lower(title)`. Індекс допомагає лише виразу, під який побудований. Те саме видно на SQLite з B-tree індексом.

In [ ]:
import random
import sqlite3
import time

random.seed(51)
conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE news (id INTEGER PRIMARY KEY, title TEXT NOT NULL, category TEXT NOT NULL)")
CATEGORIES = ["Новини", "Економіка", "Спорт", "Технології", "Культура"]
conn.executemany("INSERT INTO news (title, category) VALUES (?, ?)",
                 ((f"Новина {i}", random.choice(CATEGORIES)) for i in range(200_000)))
conn.execute("CREATE INDEX ix_news_category_lower ON news (lower(category))")


def plan(sql: str, *params) -> str:
    return " | ".join(row[-1] for row in conn.execute("EXPLAIN QUERY PLAN " + sql, params))


print("category = ?        →", plan("SELECT id FROM news WHERE category = ?", "Спорт"))
print("lower(category) = ? →", plan("SELECT id FROM news WHERE lower(category) = ?", "спорт"))

🔮 **Прогноз:** індекс є на `lower(category)`. Чому перший запит усе одно `SCAN` — перегляд усієї таблиці?

<details>
<summary>Відповідь</summary>

Для бази `category` і `lower(category)` — різні вирази. Значення в індексі — вже перетворені `lower()`, і шукати в ньому «Спорт» як є не можна. Запит і індекс мають збігатися дослівно: або індекс на `category`, або запит з `lower(category)`.
</details>

### Вправа 4. Зайві індекси

`subscriptions` з `news_hub`: `UNIQUE (chat_id, keyword)` і окремий індекс на `chat_id`. Індекс зайвий, якщо його стовпці — **початок** стовпців іншого індексу тієї ж таблиці (як телефонна книга за «прізвище, ім'я» вже впорядкована за прізвищем).

Напиши `redundant(conn, table)` → множину пар `(зайвий, той що його покриває)`. Стовпці індексу — `PRAGMA index_info(назва)` (третє поле рядка — ім'я стовпця), список індексів — `PRAGMA index_list(таблиця)` (друге поле — назва).

In [ ]:
conn.executescript("""
CREATE TABLE subscriptions (id INTEGER PRIMARY KEY, chat_id INTEGER NOT NULL, keyword TEXT NOT NULL,
                            UNIQUE (chat_id, keyword));
CREATE INDEX ix_subscriptions_chat_id ON subscriptions (chat_id);
CREATE INDEX ix_subscriptions_keyword ON subscriptions (keyword);
""")


def columns(conn: sqlite3.Connection, index: str) -> tuple[str, ...]:
    return tuple(row[2] for row in conn.execute(f"PRAGMA index_info('{index}')"))


def redundant(conn: sqlite3.Connection, table: str) -> set[tuple[str, str]]:
    # YOUR CODE HERE
    # BEGIN SOLUTION
    indexes = {row[1]: columns(conn, row[1]) for row in conn.execute(f"PRAGMA index_list('{table}')")}
    return {(small, big) for small, cols in indexes.items() for big, other in indexes.items()
            if small != big and len(cols) < len(other) and other[:len(cols)] == cols}
    # END SOLUTION


for row in conn.execute("PRAGMA index_list('subscriptions')"):
    print(row[1], columns(conn, row[1]))
print("зайві:", redundant(conn, "subscriptions"))
assert redundant(conn, "subscriptions") == {("ix_subscriptions_chat_id", "sqlite_autoindex_subscriptions_1")}
assert redundant(conn, "news") == set()
print("✅ Вправа 4 пройдена")

`ix_subscriptions_keyword` не зайвий: `keyword` — **другий** стовпець унікального індексу, а за другим стовпцем книга не впорядкована. На PostgreSQL те саме робить `db_audit.sql` уроку — і ще порівнює класи операторів (Django-індекси `*_like` не дублі).

---
## 5. Пагінація: OFFSET і keyset

`GET /api/news?skip=…` робить `ORDER BY id OFFSET skip LIMIT 50`. Щоб віддати глибоку сторінку, база читає й викидає всі попередні рядки. Keyset-пагінація просить «наступні після `id = X`» — і починає читати одразу з потрібного місця індексу.

In [ ]:
def timed(sql: str, *params) -> float:
    start = time.perf_counter()
    for _ in range(20):
        conn.execute(sql, params).fetchall()
    return (time.perf_counter() - start) / 20 * 1000


deep_offset = timed("SELECT id, title FROM news ORDER BY id LIMIT 50 OFFSET ?", 190_000)
deep_keyset = timed("SELECT id, title FROM news WHERE id > ? ORDER BY id LIMIT 50", 190_000)
print(f"OFFSET 190000: {deep_offset:.2f} мс")
print(f"id > 190000:   {deep_keyset:.3f} мс")
print(plan("SELECT id FROM news WHERE id > ? ORDER BY id LIMIT 50", 190_000))
assert deep_keyset * 10 < deep_offset

```mermaid
flowchart TD
    classDef step     fill:#263238,stroke:#90a4ae,color:#ffffff;
    classDef decision fill:#37474f,stroke:#64b5f6,color:#ffffff;
    classDef success  fill:#1b5e20,stroke:#4CAF50,color:#ffffff;
    classDef error    fill:#4e1f1f,stroke:#f44336,color:#ffffff;
    classDef warning  fill:#4a3b00,stroke:#ff9800,color:#ffffff;

    subgraph P1["запит 1: after_id = 0"]
        direction LR
        P1a["WHERE id > 0<br>LIMIT 3"] --> P1b["id 1, 2, 3"] --> P1c["next_after = 3"]
    end
    subgraph P2["запит 2: after_id = 3"]
        direction LR
        P2a["WHERE id > 3<br>LIMIT 3"] --> P2b["id 4, 5, 7<br>(6 видалено)"] --> P2c["next_after = 7"]
    end
    subgraph P3["запит 3: after_id = 7"]
        direction LR
        P3a["WHERE id > 7<br>LIMIT 3"] --> P3b["id 8"] --> P3c["менше за LIMIT:<br>кінець"]
    end
    P1 --> P2 --> P3

    class P1a,P2a,P3a,P1b,P2b,P3b step
    class P1c,P2c warning
    class P3c success
```

### Вправа 5

Напиши генератор `pages(conn, limit)`: кожен `yield` — список `id` однієї сторінки, курсор — `id` останнього рядка попередньої сторінки; зупинись, коли сторінка коротша за `limit`. Перевірка — на таблиці з «дірками» (частину новин видалено), як у справжній базі.

In [ ]:
small = sqlite3.connect(":memory:")
small.execute("CREATE TABLE news (id INTEGER PRIMARY KEY, title TEXT)")
small.executemany("INSERT INTO news (id, title) VALUES (?, ?)", ((i, f"n{i}") for i in range(1, 101)))
small.execute("DELETE FROM news WHERE id % 7 = 0")                  # «дірки» в id


def pages(conn: sqlite3.Connection, limit: int):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    after = 0
    while True:
        page = [row[0] for row in conn.execute("SELECT id FROM news WHERE id > ? ORDER BY id LIMIT ?", (after, limit))]
        if page:
            yield page
        if len(page) < limit:
            return
        after = page[-1]
    # END SOLUTION


result = list(pages(small, 10))
flat = [i for page in result for i in page]
print(len(result), "сторінок; перша:", result[0], "; остання:", result[-1])
assert flat == [row[0] for row in small.execute("SELECT id FROM news ORDER BY id")]
assert len(flat) == len(set(flat)) == 86
assert list(pages(small, 86))[-1][-1] == 100
print("✅ Вправа 5 пройдена")

🔮 **Знайди помилку:** студент написав `WHERE id >= ?`. Що побачить клієнт і що буде при `limit=1`?

<details>
<summary>Відповідь</summary>

Кожна сторінка починається з останньої новини попередньої: курсор — `id` уже показаного рядка, його треба виключити (`>`). З `limit=1` сторінка щоразу та сама — нескінченний цикл. Саме тому в перевірці вище — `len(flat) == len(set(flat))`: повтор ловиться одразу.
</details>

---
## 6. Стан і репліки

Throttle DRF у проєкті нотаток — 5 спроб входу за хвилину — рахує спроби в кеші Django. Типовий кеш — пам'ять процесу. Змоделюємо nginx, що по черзі роздає 12 спроб входу між репліками, у кожної з яких свій або спільний кеш.

### Вправа 6

Допиши `attempts(replicas, shared)` → список кодів відповіді (401 — пароль невірний, спробу пропущено; 429 — ліміт). `shared=True` — один словник-кеш на всіх (Redis), `False` — у кожної репліки свій (пам'ять процесу).

In [ ]:
RATE = 5


def allow(cache: dict[str, int], ip: str) -> bool:
    cache[ip] = cache.get(ip, 0) + 1
    return cache[ip] <= RATE


def attempts(replicas: int, shared: bool, n: int = 12, ip: str = "172.19.0.1") -> list[int]:
    # YOUR CODE HERE
    # BEGIN SOLUTION
    redis: dict[str, int] = {}
    caches = [redis] * replicas if shared else [{} for _ in range(replicas)]
    return [401 if allow(caches[i % replicas], ip) else 429 for i in range(n)]
    # END SOLUTION


def summary(codes: list[int]) -> str:
    return ", ".join(f"{code}×{codes.count(code)}" for code in sorted(set(codes)))


for replicas in (1, 2, 3):
    print(f"реплік {replicas}: пам'ять процесу → {summary(attempts(replicas, False)):<16} Redis → {summary(attempts(replicas, True))}")
assert summary(attempts(2, False)) == "401×10, 429×2"
assert summary(attempts(2, True)) == summary(attempts(1, False)) == "401×5, 429×7"
print("✅ Вправа 6 пройдена")

Модель збігається зі справжнім стеком уроку 49 (nginx, дві репліки daphne, 12 запитів `POST /api/token/`): з кешем у пам'яті — `401×10, 429×2`, з `CACHES = RedisCache` — `401×5, 429×7`. З трьома репліками в пам'яті процесу ліміт у 5 спроб узагалі не спрацьовує на 12 запитах. Звичайні тести цього не бачать: вони запускають один процес.

---
## 7. Чек-лист фінального проєкту

### Вправа 7

Напиши `check_project(root)` → словник `{пункт: True/False}`:

- `README` — є `README.md`;
- `залежності` — `requirements.txt` або `pyproject.toml`;
- `тести` — будь-де файл `test*.py`;
- `міграції` — будь-де папка `migrations`;
- `.env.example` — є, а `.env` згадано в `.gitignore` (сам файл `.env` у репозиторій не потрапляє);
- `Docker` — є `Dockerfile` і `docker-compose.yml`.

In [ ]:
def check_project(root: Path) -> dict[str, bool]:
    # YOUR CODE HERE
    # BEGIN SOLUTION
    ignore = root / ".gitignore"
    return {
        "README": (root / "README.md").is_file(),
        "залежності": (root / "requirements.txt").is_file() or (root / "pyproject.toml").is_file(),
        "тести": any(root.rglob("test*.py")),
        "міграції": any(p.is_dir() for p in root.rglob("migrations")),
        ".env.example": (root / ".env.example").is_file() and ignore.is_file()
                        and ".env" in ignore.read_text(encoding="utf-8").split(),
        "Docker": (root / "Dockerfile").is_file() and (root / "docker-compose.yml").is_file(),
    }
    # END SOLUTION


for name in ("news_hub", "crispy_notes_project"):
    report = check_project(Path(PROJECTS) / name)
    print(f"{name:<22}", "  ".join(("✅ " if ok else "❌ ") + item for item, ok in report.items()))
    assert all(report.values())
empty = Path("my_final_project")
empty.mkdir(exist_ok=True)
(empty / "main.py").write_text("print('hello')\n", encoding="utf-8")
print(f"{'my_final_project':<22}", "  ".join(("✅ " if ok else "❌ ") + item for item, ok in check_project(empty).items()))
assert not any(check_project(empty).values())
print("✅ Вправа 7 пройдена")

Обидва проєкти курсу проходять увесь чек-лист. Порожній проєкт — ні в чому, і це нормально для першого дня. План з книги: скелет з README, першим тестом і CI — **до** першої функції, а Docker і Compose — щойно з'явиться база.

Запусти `check_project` на своєму проєкті: `check_project(Path("шлях/до/проєкту"))`. CI (`.github/workflows/`) тут не перевіряємо: у курсі workflows лежать у корені репозиторію, а не в папці проєкту.

---
## Самоперевірка

1. Що означає високий fan-in і нульовий fan-out модуля?
2. Чому в циклі після ребра `db → repository` опинився й `tables`?
3. Чому індекс на `lower(category)` не допоміг запиту `WHERE category = ?`?
4. Чим keyset-пагінація краща за OFFSET, крім швидкості?
5. Чому помилку з throttle у пам'яті процесу не ловлять звичайні тести?

<details>
<summary>Відповіді</summary>

1. Фундамент: від модуля залежить багато інших, а він — ні від кого. Його інтерфейс змінювати найдорожче, тож тестів у нього має бути найбільше.
2. `repository → tables → db → repository`: `tables` лежить на шляху між двома модулями циклу, отже, теж залежить сам від себе.
3. Індекс зберігає значення `lower(category)`; вираз `category` для бази інший.
4. Сторінки не зсуваються, коли між запитами додають чи видаляють рядки; немає повторів і пропусків.
5. Тести запускають один процес — один кеш. Лічильники розходяться лише з кількома репліками.
</details>

## Далі

- Книга курсу: [Урок 51](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m6/lesson_51/) — зріз курсу, схеми баз обох проєктів, `EXPLAIN ANALYZE` на PostgreSQL (trigram, keyset, зайві індекси), граф залежностей, стан і репліки, вимоги та план фінального проєкту.
- Інструменти уроку: `architecture_audit.py`, `db_audit.sql`, `query_count.py`, `patches/news_hub_search_index.patch`.
- Урок 52 — презентація фінального проєкту.